# Marks Calculator - ECON2102 (2026, S1)

by [MachinaFantasma](https://phantomachine.github.io/)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import scipy.interpolate as interp

Purpose:

0. Clean up messy and inconsistently updated *dropped*, (late) *withdrawn*, *enroled* and/or *ECA-deferred* statuses from student records crucially drawn from three non-uniform and inconsistent sources:
    * Examinations Office Record (`EOR`) of (final-exam) attendance (may have incorrect attendance info--e.g., attend then also allowed ECA-DA!)
    * `CANVAS` assessments MArks Book record
    * Student Record (Official) -- for marks uploading

1. Get cleaned and re-weighted OMR scanner results from ``MCQ-Reweighter.ipynb``'s output (an Excel file)

2. Get all in-semester (Hurdle) assessment item grades downloaded from course LMS. This year, we pre-calculated these Hurdle items using `markscalculator-HQ+WS.ipynb`.

3. Clean up student-ID format inconsistency between LMS and OMR records (get rid of "u" prefix), and then remove the "u" again for official upload use.

    * Then merge data from these different databases matching on consistent student ID

4. Perform (possibly) some order-preserving scaling of marks distribution (no regress on any raw marks).

5. Reformat student names: Export in format that administrators need, yet another inconsistent index (student name format)!

## Specify file path(s) and names

In [ ]:
# Users can modify this!

# Specify course code
coursecode = "ECON2102"

# Data file prefix
prefix = coursecode + "_MCQ report_S1_2026 Final"

# Input data path
inpath = "data/"

# Output file name format
outfile = coursecode + "-grades"

# INPUT: Official admin sheet
officialfile = "official-form/" + coursecode + ".xlsx"

# OUTPUT: 
officialpath = "official/"
official_outfile = officialpath + coursecode + "-out.xlsx"


file_out = officialpath + outfile + ".xlsx"
file_out_csv = officialpath + outfile + ".csv"
stats_out = officialpath + outfile + "-stats" + ".xlsx"

# What stats would you like to report?
stat_au_choix = [np.mean, np.median, np.std, min, max, len]

## ANU Grade Scale

In [ ]:
def grade_scale(x):
    """Apply ANU's world-class grade scale system. 
    Take series or array x, count the number of 
    cases in each grade class."""
    # Count the number of obs in each grade class
    N = x.count()
    scales = {"N": x.between(0.0, 45., inclusive="left").sum()/N,
              "PX": x.between(45.0, 50., inclusive="left").sum()/N,
              "P": x.between(50.0, 60., inclusive="left").sum()/N,
              "CR": x.between(60.0, 70., inclusive="left").sum()/N,
              "D": x.between(70.0, 80., inclusive="left").sum()/N,
              "HD": x.between(80, 100., inclusive="both").sum()/N,
              }
    # Convert to dataframe
    df = pd.DataFrame.from_dict(dict(scales),
                                orient="index",
                                columns=["Percent",])
    return df.round(2)*100/df.sum()

## Grade-inflation-rate function

In [ ]:
# Re-weighting/scaling function - assume general logistic
def logit_weight(series=None, k=1.0, L=1.0, center=0.0, display=True):
    if series==None:
        x = np.linspace(0.0, 100, 100)
    else:
        x = series
    y = L/(1.0 + np.exp(-k*(x - center)))
    # Create weighting function interpolant
    wfun = interp.interp1d(x, y, kind="slinear")
    if display:
        plt.figure()
        plt.plot(x, y, 
                 label=r"$k$ = %1.2g, $L$= %1.2g, $x_0$=%1.2g" %(k,L,center))
        plt.legend()
        plt.show()
    return wfun

## Clean up to match student databases (3 sources!)

Three inconsistent ANU database formats. These require some matching up and checking for enrolment, withdrawal, course drop, final-exam attendance, and EAP (deferred approvals). The last two could co-exist, and this requires an extra check in the code.

1. Examination Office Registry

2. Student Registry (Official Marks)

3. CANVAS Marks Book

We need to match up 1 and 2, and map these as selection from CANVAS Marks Book!

### Data from Examination Office Registry

* May not be up to date (on late WD - withdrawals)

In [ ]:
f = "ECON2102_Semester_1_Attendance_Summary_Semester_1___End_of_Semester_2026.xlsx"
dfin = pd.read_excel(inpath + "final/" + f)
list_absentees = dfin["Student ID"][dfin["Attendance"] == "No"]
list_attendees = dfin["Student ID"][dfin["Attendance"] == "Yes"]
uid_absentees = ["u" + str(id) for id in list_absentees]
uid_attendees = ["u" + str(id) for id in list_attendees]

In [ ]:
uid_absentees

In [ ]:
# Python set - Exams Office Registry of "Enrolments"
set_EOR = set(dfin["Student ID"])

In [ ]:
print("Exams Office Database ...\n")
print("\tOfficial (Final) Enroled Student Number = %i pax"  % (len(uid_attendees)+len(uid_absentees)))

### Data from Student Registry (Official)

In [ ]:
dfs = pd.read_excel(officialfile)

bool_WD = dfs["Grade"] == "WD"
bool_enroled = (dfs["Enrolment Status"] == "Enroled")
bool_dropped = (dfs["Enrolment Status"] == "Dropped")

print("# Late Withdrawn (WD) Students = %i" % sum(bool_WD))
print("# Dropped Students = %i" % sum(bool_dropped))
print("# Enroled Students = %i" % sum(bool_enroled))

# Store IDs of types

In [ ]:
uid_official_WD = dfs.loc[bool_WD, "Emplid"]
uid_official_WD

In [ ]:
# Make Series "Emplid" int
dfs.loc[:,"Emplid"] = dfs.loc[:,"Emplid"].values.astype(int)

In [ ]:
# Python set - Student Registry Office (Official Upload Database!)
uid_official_enroled = dfs[bool_enroled]["Emplid"]
set_SRO = set(uid_official_enroled)

In [ ]:
# Sanitation (sanity) check on enrolment records!
if not set_SRO.difference(set_EOR):
    print("Quack! You lucky duck plucked from the muck ... \n")
    print("\n\t The SRO and EOR entries of enrolments agree.")
    print("\n\t Proceed to use the SRO list `uid_official_enroled`.")
else:
    print(set_SRO.difference(set_EOR))

In [ ]:
# list of official enroled (may include late WD) (int)
uid_official_enroled

In [ ]:
# list of official enroled (may include late WD) (str)
uid_enroled = ["u" + str(id) for id in uid_official_enroled]

## Final A: Import OMR report (xlsx format) $\rightarrow$ ``dfA``

* Multiple choice section of final examination.

In [ ]:
# Import as dataframe
df_mcq = pd.read_excel(inpath + "final/" + prefix +
                       "-reweighted.xlsx")

In [ ]:
# df_mcq

In [ ]:
df_mcq = df_mcq.set_index("UNum")
dfA = df_mcq.drop(index=("Answer Key"))
# dfA.reset_index()
# dfA

In [ ]:
dfA = dfA.reset_index()

In [ ]:
# Add "u" prefix to ID numbers
dfA["UNum"] = "u" + dfA["UNum"]

In [ ]:
dfA.head()

### Add dummy rows for non-sitting / DA students


**Note**:
* Actual students sitting the final exam are typically a strict subset of enrolled students.
* The official marks upload spreadsheet would contain a superset of results from final exam.
* The difference of these two sets are the "missing students".
* Here, we append the IDs of the "missing students" to the OMR records and store the marks as `NaN`s.

In [ ]:
# Get column names as a Series
dfA_keys = pd.Series(dfA.columns)

# Locate index of Series containing "UNum"
index_UNum = dfA_keys.index[dfA_keys=="UNum"]

In [ ]:
# Expand rows by length of uid_absentees
N_abs = len(uid_absentees)
N_dfA = len(dfA)
dfAex = dfA.reindex(range(N_dfA + N_abs))

In [ ]:
# Append "missing students" (IDs) to dfA
dfAex.iloc[N_dfA : N_dfA + N_abs, index_UNum] = uid_absentees

In [ ]:
# Overwrite original dfA with expanded list of dummies
dfA = dfAex.copy().fillna(0.0)

## Final B: Import from Grader's Excel Report $\rightarrow$ `dfB`

In [ ]:
dfB_temp = pd.read_excel(inpath + "final/" + "2026-06-06-ECON2102-Final-B-IRIS.xlsx", sheet_name="Totals")
dfB_temp.head()

In [ ]:
pp_nuisance = dfB_temp.iloc[0]["LastName"]
pp_nuisance

In [ ]:
# Drop these records - index by "LastName" - junk from CANVAS
index_drops = [pp_nuisance, "Student"]

In [ ]:
dfb = dfB_temp.set_index("LastName")

In [ ]:
dfB = dfb.drop(index=index_drops).reset_index()

In [ ]:
set_CANVAS = set(dfB["SIS User ID"].values.tolist())

In [ ]:
# Add "u" prefix
set_uSRO = ["u" + str(idx) for idx in set_SRO]

In [ ]:
# Nuisance student ID - not in official SRO list
uid_CANVAS_drop = list(set_CANVAS.difference(set_uSRO))
uid_CANVAS_drop

In [ ]:
# Drop the nuisance student ID by named index
dfB = dfB.set_index("SIS User ID").drop(index=uid_CANVAS_drop).reset_index()
dfB = dfB.fillna(0.0)

## Hurdles: Assessments from Course Web ``CANVAS`` $\rightarrow$ `dfH`

**Hurdles** marks were previously computed using `Jupyter` notebook [markscalculator-HQ+WS.ipynb](./markscalculator-HQ+WS.ipynb) in this same working directory. 
* For static `HTML` preview of this notebook, [see this file here](./markscalculator-HQ+WS.html).

Hurdle Assessments as specified in ANU Programs and Courses:

* Top-5 of [Weekly Workshop Quizzes](https://programsandcourses.anu.edu.au/course/ECON2102/First%20Semester/3640#assessmenttask-2) (WQ) (20%)

* Top-3 of [Bi-weekly Hurdle Quiz](https://programsandcourses.anu.edu.au/course/ECON2102/First%20Semester/3640#assessmenttask-1) (HQ) (40%)

In [ ]:
# Import CANVAS Marks Book as dataframe
dfW = pd.read_csv(inpath + "hurdles/" +
                  "ECON2102-hurdles-out.csv")

In [ ]:
dfW.head()

In [ ]:
# You can see the number of keys, including that of assessment items
dfW.keys()

In [ ]:
# CANVAS student data, important to keep for uploading to CANVAS again!
keys_canvas = ['LastName', 'FirstName', 'ID', 'SIS User ID', 'SIS Login ID',
               'Integration ID', 'Section', ]

# Pick out just the aggregated assessment marks (Final exam part B pre-loaded to CANVAS)
keys_assess = ['All Hurdles (Max 60)']

In [ ]:
df = dfW[keys_canvas + keys_assess]

In [ ]:
# Drop junk rows from CANVAS, affects dfH too
dfH = (df.set_index("LastName")).drop(index=index_drops).reset_index()

# Again, drop nuisance student ID from dfH
dfH = dfH.set_index("SIS User ID").drop(index=uid_CANVAS_drop).reset_index()

In [ ]:
# # Drop rows 0, 1, and 31 from previous view manually
# df = dfW[keys_canvas + keys_assess].drop([0, 1, 31]).set_index("SIS User ID")
# df

## Merge dataframes ``dfH`` (CANVAS: Hurdles), ``dfA`` (Final A), ``dfB`` (Final B)

In [ ]:
dfB.set_index(keys_canvas)

In [ ]:
dfh = dfH.set_index(keys_canvas)
dfb = dfB.set_index(keys_canvas)
dHB = dfh.join(dfb, on=keys_canvas)
dHB

In [ ]:
# dfB and dfA, go by Uni ID (SQL left merge)
dg_temp = dHB.reset_index().merge(dfA[["UNum", "Total"]], 
                             how='right', left_on="SIS User ID", right_on="UNum")

In [ ]:
dg = dg_temp.drop(columns="UNum").rename(columns={"Total": "A"})
dg.head()

In [ ]:
dg.reset_index()

In [ ]:
# Final Exam Marks
# A ~ 50
# B1 ~ 25
# B2 ~ 25
dg.loc[:, "Final"] = dg["B1"] + dg["B2"] + dg["A"]

## Consistency checks on student status

### Withdrawn students (`WD`)

In [ ]:
# Add enrolment status
dg["Enrolment"] = "Enroled"

In [ ]:
# List of str IDs of WD students
uid_WD = [ "u" + str(id) for id in uid_official_WD ]
uid_WD

In [ ]:
# Update to true status as WD
dg = dg.set_index("SIS User ID")
dg.loc[uid_WD, "Enrolment"] = "WD"

### ECA - Deferred Exams Approved (`DA`)

In [ ]:
# Add Exam Status
dg["ECA Status"] = "None"

In [ ]:
# Manually compiled DA approvals - from individual emails

uid_DA = pd.read_excel(
    inpath + "ECA/" + "ECON2102-ECA-DA-approvals.xlsx")
uid_DA = list(uid_DA)

In [ ]:
# Store approved DA status
dg.loc[uid_DA, "ECA Status"] = "DA approved"

### Final Attendance

In [ ]:
# Store sttendance status: N/A, No, or Yes
dg["Final Attendance"] = "N/A"
dg.loc[uid_absentees, "Final Attendance"] = "No"
dg.loc[uid_attendees, "Final Attendance"] = "Yes"

### ECA - Assessment Weight Adjustments

In [ ]:
# Late adjustments

dg["ECA Assessment Adjustments"] = "No"

uid_ECAadjust = pd.read_excel(
    inpath + "ECA/" + "ECON2102-ECA-Assessment-Weights-Adjustment.xlsx")
uid_ECAadjust = list(uid_ECAadjust)

dg.loc[uid_ECAadjust, "ECA Assessment Adjustments"] = "Shift 5 percent: Final to Hurdles"

## Course mark

Assessment structure

*  Hurdle Assessments (60% of course)

    * Task 1: Top-3 of [Bi-weekly Hurdle Quiz](https://programsandcourses.anu.edu.au/course/ECON2102/First%20Semester/3640#assessmenttask-1) (HQ) (40%)

    * Task 2: op-5 of [Weekly Workshop Quizzes](https://programsandcourses.anu.edu.au/course/ECON2102/First%20Semester/3640#assessmenttask-2) (WQ) (20%)


* Task 3: [Final Examination](https://programsandcourses.anu.edu.au/course/ECON2102/First%20Semester/3640#assessmenttask-3) - (40% of course)
    * Final A - 50% of Final
    * Final B1 - 25% of Final
    * Final B2 - 25% of Final

In [ ]:
dg.loc[:, "Course"] = 0.4*dg.loc[:, "Final"] + dg.loc[:, "All Hurdles (Max 60)"]

In [ ]:
# ECA Assessment adjustment (ex post) - Shift 0.05 from Final to All Hurdles
dg.loc[uid_ECAadjust, "Course"] = 0.35*dg.loc[uid_ECAadjust, "Final"] + 1.05*dg.loc[uid_ECAadjust, "All Hurdles (Max 60)"]

### Summary statistics

Statistics are calculated based on only those who completed the final exam.

In [ ]:
# Distribution relevant to only enroled AND non-DAs
dfd = dg[dg["Enrolment"]=="Enroled"]
dfd = dfd[dfd["ECA Status"] == "None"]

print("Check: attendance of final = %i pax" % len(dfd))

In [ ]:
# Rounding
dfd.loc[:,"RoundUp"] = np.round(dfd.loc[:,"Course"].values)

In [ ]:
# Only those who completed the course (dropna)
course_mark = dfd.RoundUp.dropna()

In [ ]:
# Indicator functions - borders of key grades
border_CR = 1*course_mark.between(59, 60, inclusive="left")
border_D = 1*course_mark.between(69, 70, inclusive="both")
border_HD = 1*course_mark.between(79, 80, inclusive="both")

# Do relevant bump ups!
dfd.loc[:,"BorderUp"] = dfd.loc[:, "RoundUp"] + border_CR + border_D + border_HD

In [ ]:
# Raw distribution statistics
stat1 = dfd.BorderUp.apply(stat_au_choix).round(2)
df_stat1 = pd.DataFrame.from_dict(dict(stat1),
                                  orient="index",
                                  columns=["value",])
df_stat1

In [ ]:
# Raw grade shares
grades_stats = grade_scale(dfd.BorderUp)
grades_stats

In [ ]:
ax = dfd.BorderUp.hist(bins=20)
dfd.BorderUp.plot.kde(ax=ax, secondary_y=True, title="Course Marks")

## Re-scale distribution 

Using the weighting function $1 + w(x)$ where the rate function $w$ is logistic, as shown below:

In [ ]:
# Instance of the function, an object
w = logit_weight(k=0.0, L=0.0, center=100)

In [ ]:
# Example: value of weighting function at a point
w(44.)

In [ ]:
X = dfd["Course"].dropna()
dfd.loc[:,"Scaled"] = np.round(X*(1.0 + w(X.values.tolist())))

# Indicator functions - borders of key grades
border_CR = 1*dfd["Scaled"].between(59, 60, inclusive="left")
border_D = 1*dfd["Scaled"].between(69, 70, inclusive="left")
border_HD = 1*dfd["Scaled"].between(79, 80, inclusive="left")

# Do relevant bump ups!
dfd.loc[:,"Official"] = dfd.loc[:,"Scaled"] + border_CR + border_D + border_HD

# Borderline -> Pass!
dfd.loc[dfd.Official.between(40, 50, inclusive="left"), 'Official'] = 50.0

In [ ]:
false_bumps = dfd.index[dfd.Scaled.between(0, 44, inclusive="left")].to_list()
dfd.loc[false_bumps, "Scaled"] = dfd.loc[false_bumps]["BorderUp"]
dfd.loc[false_bumps, "Official"] = dfd.loc[false_bumps]["BorderUp"]

In [ ]:
dfd.sort_values(by=["Official",], ascending=True)

In [ ]:
# Borderline -> Pass!
dfd.loc[dfd.Official.between(40, 50, inclusive="left"), 'Official'] = 50.0

In [ ]:
# Raw grade shares
grades_stats = grade_scale(dfd["Official"])
grades_stats

In [ ]:
# Raw distribution statistics
stat2 = dfd.Official.dropna().apply(stat_au_choix).round(2)
df_stat2 = pd.DataFrame.from_dict(dict(stat2),
                                  orient="index",
                                  columns=["value",])
df_stat2

In [ ]:
ax = dfd.Official.hist(bins=60)
dfd.Official.plot.kde(ax=ax, secondary_y=True, title="Official Course Marks");

## Convert to admin format

In [ ]:
# Official spreadsheet format
dfo = pd.read_excel(officialfile)
dfo.Emplid = dfo["Emplid"].dropna().astype(int)
dfo.loc[:, "SIS User ID"] = ["u" + str(id) for id in dfo.Emplid.values]
dfo = dfo.set_index("SIS User ID")
dfo

In [ ]:
official_mark = "3630 (mark)"

In [ ]:
dfo[official_mark] = 0.0
dfo

In [ ]:
# Possible that attendee of final can still get DA! 
set_attend_DA = set(uid_DA).intersection(set(uid_attendees))
set_attend_DA

In [ ]:
# Minus attend+DA types from attendees!
uid_attendees_truelist = list(set(uid_attendees).difference(set_attend_DA))

In [ ]:
# Default enroled get base "Course" mark (unscaled)
dfo.loc[uid_enroled, official_mark] = dg.loc[uid_enroled, "Course"]

# Some enroled do sit the final, so update as "Official" (scaled)
dfo.loc[uid_attendees_truelist,
           official_mark] = dfd.loc[uid_attendees_truelist, "Official"]

In [ ]:
# cut turns series into categorical variable
dfo.loc[:, "Grade"] = pd.cut(dfo[official_mark],
                     bins=[0., 45., 50., 60., 70., 80., 100.1],
                     right=False,
                     labels=['N', 'PX', 'P', 'CR', 'D', 'HD']
                     )

# Series Grade is 'categorical' dtype. Make it 'str' dtype!
dfo.loc[:, "Grade"] = dfo["Grade"].astype('str')

In [ ]:
# Some enroled are also late "WD", so update as "WD"
dfo.loc[uid_WD, "Grade"] = "WD"
dfo.loc[uid_WD, official_mark] = np.nan

# Approved DA (ECA)
dfo.loc[uid_DA, "Grade"] = "DA"
dfo.loc[uid_DA, official_mark] = np.nan

In [ ]:
# Dropped
bool_dropped = dfo["Enrolment Status"] == "Dropped"
dfo.loc[bool_dropped, "Grade"]  = np.nan

In [ ]:
dg.keys()

In [ ]:
dfo.keys()

In [ ]:
# Add more info from dg - Final Exam Attendance
dfo.loc[:, "Sat Final"] = "No"  # pre-populate
dfo.loc[uid_attendees_truelist,
           "Sat Final"] = dg.loc[uid_attendees_truelist, "Final Attendance"] 

dfo.loc[uid_WD,
           "Late WD"] = dg.loc[uid_WD, "Enrolment"]

dfo.loc[uid_DA,
           "ECA Status"] = dg.loc[uid_DA, "ECA Status"]

dfo.loc[uid_ECAadjust,
        "ECA Assessment Adjustments"] = dg.loc[uid_ECAadjust, "ECA Assessment Adjustments"]

In [ ]:
# Official report - excel sheet
dfo.to_excel(official_outfile)
dfo

## Mailist for HD / D letter

In [ ]:
dfo.loc[dfo['Grade'] == "N"]

In [ ]:
dfo.loc[dfo['Grade'] == "P"]

In [ ]:
dfo.loc[dfo['Grade'] == "CR"]

In [ ]:
dfo.loc[dfo['Grade'] == "D"]

In [ ]:
dfo.loc[dfo['Grade'] == "HD"]

### HD Honours List

In [ ]:
HD_list = dfo.loc[dfo['Grade'] == "HD"].index
mailist = [ str(value)+"@anu.edu.au" for value in HD_list ]
mailist

In [ ]:
pd.Series(mailist).to_excel("congrats-HD-mailist.xlsx")

### Top-5% Honours List

In [ ]:
top_5_percent = dfd[dfd['Official'] >= dfd['Official'].quantile(0.95)].index
top_5_percent
mailist = [str(value)+"@anu.edu.au" for value in top_5_percent]
mailist

In [ ]:
pd.Series(mailist).to_excel("congrats-Top5percent-mailist.xlsx")

## Double check ex-post ECA Assessment Weight Adjustments Cases

In [ ]:
dfo.loc[uid_ECAadjust]

In [ ]:
dfd.loc[uid_ECAadjust].T